# **FRAMEWORK V7: NOTEBOOK DE EXTRACCIÓN DE DATOS CRUDOS**

CAPA - INFORMACIÓN PERCEPCIÓN



## **M0. Configuración General**

In [1]:
import pandas as pd
import requests
import unicodedata

print("="*70)
print("FRAMEWORK V7")
print("CAPA DE PERCEPCIÓN")
print("INGESTA DE DATOS")
print("="*70)

# ============================================================
# DATASETS OFICIALES
# ============================================================

ID_IRCA = "j2wc-9pkj"

ID_CONSUMO = "nxt2-39c3"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# ============================================================
# FUNCIÓN PARA DESCARGAR DATASETS
# ============================================================

def descargar_dataset(id_dataset, nombre_archivo):

    url = f"https://www.datos.gov.co/resource/{id_dataset}.json"

    print(f"\nDescargando {nombre_archivo}...")

    r = requests.get(
        url,
        params={"$limit":50000},
        headers=headers
    )

    if r.status_code == 200:

        df = pd.DataFrame(r.json())

        df.to_excel(
            nombre_archivo,
            index=False
        )

        print("OK")

        print("Registros:",len(df))

        print("Columnas:",len(df.columns))

        return df

    else:

        print("Error:",r.status_code)

        return None


# ============================================================
# DESCARGA
# ============================================================

df_irca = descargar_dataset(

    ID_IRCA,

    "Percepcion_Indice_Riesgo_Agua.xlsx"

)

df_consumo = descargar_dataset(

    ID_CONSUMO,

    "Calidad_Agua_Consumo_Humano.xlsx"

)

FRAMEWORK V7
CAPA DE PERCEPCIÓN
INGESTA DE DATOS

Descargando Percepcion_Indice_Riesgo_Agua.xlsx...
OK
Registros: 185
Columnas: 10

Descargando Calidad_Agua_Consumo_Humano.xlsx...
OK
Registros: 19160
Columnas: 11


## **M1. Definición de la Fuente Oficial**

Los IDs de los datasets oficiales se definen en la sección M0 y la descarga inicial de los datos (`df_irca` y `df_consumo`) también se realiza allí.

## **M2. Extracción y Procesamiento de Datos**

In [2]:
print("\n")
print("="*70)
print("COLUMNAS DATASET PRINCIPAL")
print("="*70)
for c in df_consumo.columns:
    print(c)

print("\n")
print("="*70)
print("COLUMNAS DATASET VALIDACIÓN")
print("="*70)
for c in df_irca.columns:
    print(c)



COLUMNAS DATASET PRINCIPAL
departamentocodigo
departamento
municipiocodigo
municipio
a_o
irca
nivel_de_riesgo
ircaurbano
nivel_de_riesgo_urbano
ircarural
nivel_de_riesgo_rural


COLUMNAS DATASET VALIDACIÓN
fecha_de_toma_de_muestra
clasificacion_de_la_muestra
punto_de_muestreo
tipo_de_muestra
ph
cloro_residual
coliformes_totales
coliformes_fecales
t_cnica_empleada_ct_cf
resultados_del_irca_por


In [3]:
# ============================================================
# NORMALIZACIÓN DE TEXTO
# ============================================================

def normalizar(texto):
    if pd.isna(texto):
        return texto
    texto = str(texto).upper()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return texto.strip()


# ============================================================
# NORMALIZAR MUNICIPIOS
# ============================================================

df_consumo["municipio_norm"] = (
    df_consumo["municipio"]
    .apply(normalizar)
)

print(df_consumo["municipio_norm"].unique())

['#TODOS' 'BOGOTA, D.C.' 'CARTAGENA DE INDIAS' ... 'CARURU' 'TARAIRA'
 'PUERTO CARRENO']


In [4]:
print("="*70)
print("MUNICIPIOS DISPONIBLES")
print("="*70)

municipios = (
    df_consumo["municipio"]
    .dropna()
    .sort_values()
    .unique()
)

for m in municipios:
    print(m)

print()
print("Total municipios:", len(municipios))

MUNICIPIOS DISPONIBLES
#TODOS
Abejorral
Abriaquí
Acacías
Acandí
Acevedo
Achí
Agrado
Agua de Dios
Aguachica
Aguada
Aguadas
Aguazul
Agustín Codazzi
Aipe
Albania
Albán
Alcalá
Aldana
Alejandría
Algarrobo
Algeciras
Almaguer
Almeida
Alpujarra
Altamira
Alto Baudó
Altos del Rosario
Alvarado
Amagá
Amalfi
Ambalema
Anapoima
Ancuya
Andalucía
Andes
Angelópolis
Angostura
Anolaima
Anorí
Anserma
Ansermanuevo
Anzoátegui
Anzá
Apartadó
Apulo
Apía
Aquitania
Aracataca
Aranzazu
Aratoca
Arauca
Arauquita
Arbeláez
Arboleda
Arboledas
Arboletes
Arcabuco
Arenal
Argelia
Ariguaní
Arjona
Armenia
Armero
Arroyohondo
Astrea
Ataco
Atrato
Ayapel
Bagadó
Bahía Solano
Bajo Baudó
Balboa
Baranoa
Baraya
Barbacoas
Barbosa
Barichara
Barranca de Upía
Barrancabermeja
Barrancas
Barranco de Loba
Barrancominas
Barranquilla
Becerril
Belalcázar
Bello
Belmira
Beltrán
Belén
Belén de Los Andaquíes
Belén de Umbría
Berbeo
Betania
Betulia
Betéitiva
Bituima
Boavita
Bochalema
Bogotá, D.C.
Bojacá
Bojayá
Bolívar
Bosconia
Boyacá
Briceño
Bucaraman

In [5]:
print(df_consumo[
    df_consumo["municipio"].str.contains("Bog", case=False, na=False)
]["municipio"].unique())

municipios_objetivo = [
    "Villapinzón",
    "Tocancipá",
    "Chía",
    "Bogotá, D.C."
]

df_percepcion = df_consumo[
    df_consumo["municipio"].isin(municipios_objetivo)
].copy()

['Bogotá, D.C.']


In [6]:
print("="*70)
print("COLUMNAS DEL DATASET FILTRADO")
print("="*70)
print(df_percepcion.columns.tolist())

print()
print("="*70)
print("PRIMERAS Y ÚLTIMAS FILAS DEL DATASET FILTRADO")
print("="*70)
print(df_percepcion.head())
print()
print(df_percepcion.tail())

print()
print("="*70)
print("DIMENSIONES DEL DATASET FILTRADO")
print("="*70)
print(df_percepcion.shape)

print()
print("="*70)
print("VALORES NULOS EN EL DATASET FILTRADO")
print("="*70)
print(df_percepcion.isnull().sum())

print()
print("="*70)
print("REGISTROS DUPLICADOS EN EL DATASET FILTRADO")
print("="*70)
print(df_percepcion.duplicated().sum())

COLUMNAS DEL DATASET FILTRADO
['departamentocodigo', 'departamento', 'municipiocodigo', 'municipio', 'a_o', 'irca', 'nivel_de_riesgo', 'ircaurbano', 'nivel_de_riesgo_urbano', 'ircarural', 'nivel_de_riesgo_rural', 'municipio_norm']

PRIMERAS Y ÚLTIMAS FILAS DEL DATASET FILTRADO
     departamentocodigo  departamento municipiocodigo     municipio   a_o  \
32                   11  Bogotá, D.C.           11001  Bogotá, D.C.  2024   
330                  25  Cundinamarca           25175          Chía  2024   
412                  25  Cundinamarca           25817     Tocancipá  2024   
422                  25  Cundinamarca           25873   Villapinzón  2024   
3541                 11  Bogotá, D.C.           11001  Bogotá, D.C.  2007   

     irca nivel_de_riesgo ircaurbano nivel_de_riesgo_urbano ircarural  \
32    3.6      Sin riesgo        1.1             Sin riesgo      14.8   
330   3.6      Sin riesgo        3.8             Sin riesgo       3.5   
412   0.3      Sin riesgo          0    

In [7]:
df_percepcion = df_percepcion[
[
    "municipio",
    "a_o",
    "irca",
    "nivel_de_riesgo"
]].copy()

df_percepcion = df_percepcion.sort_values(
    by=["municipio","a_o"]
).reset_index(drop=True)

df_percepcion["Nodo"] = df_percepcion["municipio"]

# ============================================================
# SELECCIÓN DE VARIABLES DEL FRAMEWORK
# ============================================================

df_percepcion = df_percepcion[
    [
        "Nodo",
        "municipio",
        "a_o",
        "irca",
        "nivel_de_riesgo"
    ]
].copy()

In [ ]:
print("="*70)
print("VALORES NULOS DESPUÉS DE PROCESAMIENTO")
print("="*70)
print(df_percepcion.isnull().sum())

print()
print("="*70)
print("REGISTROS DUPLICADOS DESPUÉS DE PROCESAMIENTO")
print("="*70)
print(df_percepcion.duplicated().sum())

print()
print("="*70)
print("ESTADÍSTICAS DESCRIPTIVAS")
print("="*70)
print(df_percepcion.describe(include="all"))

print()
print("="*70)
print("NIVELES DE RIESGO - CONTEO")
print("="*70)
print(df_percepcion["nivel_de_riesgo"].value_counts())

print()
print("="*70)
print("MUNICIPIOS PROCESADOS - CONTEO")
print("="*70)
print(df_percepcion["municipio"].value_counts())

## **M3. Reporte de Auditoría**

In [8]:
print("="*70)
print("GENERANDO REPORTE DE AUDITORÍA")
print("="*70)

# Prepare data for the audit report
audit_data = {
    "Métrica": [
        "Dimensiones (filas, columnas)",
        "Total nulos",
        "Total duplicados",
        "Conteo Nivel de Riesgo",
        "Conteo por Municipio"
    ],
    "Valor": [
        str(df_percepcion.shape),
        df_percepcion.isnull().sum().sum(),
        df_percepcion.duplicated().sum(),
        df_percepcion["nivel_de_riesgo"].value_counts().to_string(),
        df_percepcion["municipio"].value_counts().to_string()
    ]
}

df_audit = pd.DataFrame(audit_data)

# Export the audit report to Excel
df_audit.to_excel("Auditoria_Capa_Percepcion.xlsx", index=False)

print("Archivo de auditoría generado correctamente: Auditoria_Capa_Percepcion.xlsx")

GENERANDO REPORTE DE AUDITORÍA
Archivo de auditoría generado correctamente: Auditoria_Capa_Percepcion.xlsx


## **M4. Exportación de Resultados**

In [9]:
# ============================================================
# EXPORTAR CAPA
# ============================================================

df_percepcion.to_excel(
    "01_Capa_Percepcion_V1.xlsx",
    index=False
)

print()
print("Archivo generado correctamente")
print("01_Capa_Percepcion_V1.xlsx")


Archivo generado correctamente
01_Capa_Percepcion_V1.xlsx


## **M5. Resumen Final**

El proceso de ingesta y procesamiento de datos para la capa de percepción ciudadana ha sido completado. Se han generado los archivos '01_Capa_Percepcion_V1.xlsx' y 'Auditoria_Capa_Percepcion.xlsx'.

FRAMEWORK V7

CAPA 6 - INFORMACIÓN PERCEPCIÓN

M1 – Ingesta de Datos

La dimensión de percepción ciudadana fue construida mediante el análisis de la Calidad del Agua para Consumo Humano y el Índice de Riesgo de la Calidad del Agua (IRCA). Estos indicadores actúan como proxies técnicos de la percepción social, dado que los niveles de riesgo sanitario reportados por el sistema de salud pública son el determinante primario de la confianza ciudadana y la conflictividad social en la cuenca. La estructura de integración mantiene la consistencia espacial mediante el identificador de nodo (nodo_id), permitiendo una correlación directa entre el riesgo sanitario y las dinámicas climáticas

In [ ]:
import pandas as pd
import requests
import unicodedata

print("="*70)
print("FRAMEWORK V7")
print("CAPA DE PERCEPCIÓN")
print("INGESTA DE DATOS")
print("="*70)

# ============================================================
# DATASETS OFICIALES
# ============================================================

ID_IRCA = "j2wc-9pkj"

ID_CONSUMO = "nxt2-39c3"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# ============================================================
# FUNCIÓN PARA DESCARGAR DATASETS
# ============================================================

def descargar_dataset(id_dataset, nombre_archivo):

    url = f"https://www.datos.gov.co/resource/{id_dataset}.json"

    print(f"\nDescargando {nombre_archivo}...")

    r = requests.get(
        url,
        params={"$limit":50000},
        headers=headers
    )

    if r.status_code == 200:

        df = pd.DataFrame(r.json())

        df.to_excel(
            nombre_archivo,
            index=False
        )

        print("OK")

        print("Registros:",len(df))

        print("Columnas:",len(df.columns))

        return df

    else:

        print("Error:",r.status_code)

        return None


# ============================================================
# DESCARGA
# ============================================================

df_irca = descargar_dataset(

    ID_IRCA,

    "Percepcion_Indice_Riesgo_Agua.xlsx"

)

df_consumo = descargar_dataset(

    ID_CONSUMO,

    "Calidad_Agua_Consumo_Humano.xlsx"

)

FRAMEWORK V7
CAPA DE PERCEPCIÓN
INGESTA DE DATOS

Descargando Percepcion_Indice_Riesgo_Agua.xlsx...
OK
Registros: 185
Columnas: 10

Descargando Calidad_Agua_Consumo_Humano.xlsx...
OK
Registros: 19160
Columnas: 11


In [ ]:
print("\n")

print("="*70)

print("COLUMNAS DATASET PRINCIPAL")

print("="*70)

for c in df_consumo.columns:

    print(c)

print("\n")

print("="*70)

print("COLUMNAS DATASET VALIDACIÓN")

print("="*70)

for c in df_irca.columns:

    print(c)



COLUMNAS DATASET PRINCIPAL
departamentocodigo
departamento
municipiocodigo
municipio
a_o
irca
nivel_de_riesgo
ircaurbano
nivel_de_riesgo_urbano
ircarural
nivel_de_riesgo_rural


COLUMNAS DATASET VALIDACIÓN
fecha_de_toma_de_muestra
clasificacion_de_la_muestra
punto_de_muestreo
tipo_de_muestra
ph
cloro_residual
coliformes_totales
coliformes_fecales
t_cnica_empleada_ct_cf
resultados_del_irca_por


In [ ]:
# ============================================================
# NORMALIZACIÓN DE TEXTO
# ============================================================

def normalizar(texto):

    if pd.isna(texto):

        return texto

    texto = str(texto).upper()

    texto = ''.join(

        c for c in unicodedata.normalize('NFD', texto)

        if unicodedata.category(c) != 'Mn'

    )

    return texto.strip()


# ============================================================
# NORMALIZAR MUNICIPIOS
# ============================================================

df_consumo["municipio_norm"] = (

    df_consumo["municipio"]

    .apply(normalizar)

)

print(df_consumo["municipio_norm"].unique())

['#TODOS' 'BOGOTA, D.C.' 'CARTAGENA DE INDIAS' ... 'CARURU' 'TARAIRA'
 'PUERTO CARRENO']


In [ ]:
print("="*70)
print("MUNICIPIOS DISPONIBLES")
print("="*70)

municipios = (
    df_consumo["municipio"]
    .dropna()
    .sort_values()
    .unique()
)

for m in municipios:
    print(m)

print()
print("Total municipios:", len(municipios))

MUNICIPIOS DISPONIBLES
#TODOS
Abejorral
Abriaquí
Acacías
Acandí
Acevedo
Achí
Agrado
Agua de Dios
Aguachica
Aguada
Aguadas
Aguazul
Agustín Codazzi
Aipe
Albania
Albán
Alcalá
Aldana
Alejandría
Algarrobo
Algeciras
Almaguer
Almeida
Alpujarra
Altamira
Alto Baudó
Altos del Rosario
Alvarado
Amagá
Amalfi
Ambalema
Anapoima
Ancuya
Andalucía
Andes
Angelópolis
Angostura
Anolaima
Anorí
Anserma
Ansermanuevo
Anzoátegui
Anzá
Apartadó
Apulo
Apía
Aquitania
Aracataca
Aranzazu
Aratoca
Arauca
Arauquita
Arbeláez
Arboleda
Arboledas
Arboletes
Arcabuco
Arenal
Argelia
Ariguaní
Arjona
Armenia
Armero
Arroyohondo
Astrea
Ataco
Atrato
Ayapel
Bagadó
Bahía Solano
Bajo Baudó
Balboa
Baranoa
Baraya
Barbacoas
Barbosa
Barichara
Barranca de Upía
Barrancabermeja
Barrancas
Barranco de Loba
Barrancominas
Barranquilla
Becerril
Belalcázar
Bello
Belmira
Beltrán
Belén
Belén de Los Andaquíes
Belén de Umbría
Berbeo
Betania
Betulia
Betéitiva
Bituima
Boavita
Bochalema
Bogotá, D.C.
Bojacá
Bojayá
Bolívar
Bosconia
Boyacá
Briceño
Bucaraman

In [ ]:
print(df_consumo[
    df_consumo["municipio"].str.contains("Bog", case=False, na=False)
]["municipio"].unique())

['Bogotá, D.C.']


In [ ]:
municipios_objetivo = [
    "Villapinzón",
    "Tocancipá",
    "Chía",
    "Bogotá, D.C."
]

df_percepcion = df_consumo[
    df_consumo["municipio"].isin(municipios_objetivo)
].copy()

In [ ]:
print("="*70)
print("COLUMNAS DEL DATASET FILTRADO")
print("="*70)

print(df_percepcion.columns.tolist())

COLUMNAS DEL DATASET FILTRADO
['departamentocodigo', 'departamento', 'municipiocodigo', 'municipio', 'a_o', 'irca', 'nivel_de_riesgo', 'ircaurbano', 'nivel_de_riesgo_urbano', 'ircarural', 'nivel_de_riesgo_rural', 'municipio_norm']


In [ ]:
print(df_percepcion.head())

print()

print(df_percepcion.tail())

     departamentocodigo  departamento municipiocodigo     municipio   a_o  \
32                   11  Bogotá, D.C.           11001  Bogotá, D.C.  2024   
330                  25  Cundinamarca           25175          Chía  2024   
412                  25  Cundinamarca           25817     Tocancipá  2024   
422                  25  Cundinamarca           25873   Villapinzón  2024   
3541                 11  Bogotá, D.C.           11001  Bogotá, D.C.  2007   

     irca nivel_de_riesgo ircaurbano nivel_de_riesgo_urbano ircarural  \
32    3.6      Sin riesgo        1.1             Sin riesgo      14.8   
330   3.6      Sin riesgo        3.8             Sin riesgo       3.5   
412   0.3      Sin riesgo          0             Sin riesgo       0.3   
422   2.9      Sin riesgo        1.9             Sin riesgo       5.2   
3541  0.1      Sin riesgo        0.1             Sin riesgo       1.0   

     nivel_de_riesgo_rural municipio_norm  
32            Riesgo medio   BOGOTA, D.C.  
330       

In [ ]:
print("="*70)
print("DIMENSIONES")
print("="*70)

print(df_percepcion.shape)

DIMENSIONES
(72, 12)


In [ ]:
print("="*70)
print("VALORES NULOS")
print("="*70)

print(df_percepcion.isnull().sum())

VALORES NULOS
departamentocodigo        0
departamento              0
municipiocodigo           0
municipio                 0
a_o                       0
irca                      0
nivel_de_riesgo           0
ircaurbano                0
nivel_de_riesgo_urbano    0
ircarural                 0
nivel_de_riesgo_rural     0
municipio_norm            0
dtype: int64


In [ ]:
print("="*70)
print("DUPLICADOS")
print("="*70)

print(df_percepcion.duplicated().sum())

DUPLICADOS
0


In [ ]:
df_percepcion = df_percepcion[
[
    "municipio",
    "a_o",
    "irca",
    "nivel_de_riesgo"
]].copy()

In [ ]:
df_percepcion = df_percepcion.sort_values(
    by=["municipio","a_o"]
).reset_index(drop=True)

In [ ]:
df_percepcion["Nodo"] = df_percepcion["municipio"]

In [ ]:
# ============================================================
# SELECCIÓN DE VARIABLES DEL FRAMEWORK
# ============================================================

df_percepcion = df_percepcion[
    [
        "Nodo",
        "municipio",
        "a_o",
        "irca",
        "nivel_de_riesgo"
    ]
].copy()

In [ ]:
print("="*70)
print("VALORES NULOS")
print("="*70)

print(df_percepcion.isnull().sum())

VALORES NULOS
Nodo               0
municipio          0
a_o                0
irca               0
nivel_de_riesgo    0
dtype: int64


In [ ]:
print("="*70)
print("REGISTROS DUPLICADOS")
print("="*70)

print(df_percepcion.duplicated().sum())

REGISTROS DUPLICADOS
0


In [ ]:
print("="*70)
print("ESTADÍSTICAS")
print("="*70)

print(df_percepcion.describe(include="all"))

ESTADÍSTICAS
                Nodo     municipio   a_o irca nivel_de_riesgo
count             72            72    72   72              72
unique             4             4    18   48               3
top     Bogotá, D.C.  Bogotá, D.C.  2007  0.0      Sin riesgo
freq              18            18     4    7              52


In [ ]:
print("="*70)
print("NIVELES DE RIESGO")
print("="*70)

print(df_percepcion["nivel_de_riesgo"].value_counts())

NIVELES DE RIESGO
nivel_de_riesgo
Sin riesgo      52
Riesgo bajo     18
Riesgo medio     2
Name: count, dtype: int64


In [ ]:
print("="*70)
print("MUNICIPIOS")
print("="*70)

print(df_percepcion["municipio"].value_counts())

MUNICIPIOS
municipio
Bogotá, D.C.    18
Chía            18
Tocancipá       18
Villapinzón     18
Name: count, dtype: int64


In [ ]:
# ============================================================
# EXPORTAR CAPA
# ============================================================

df_percepcion.to_excel(
    "01_Capa_Percepcion_V1.xlsx",
    index=False
)

print()
print("Archivo generado correctamente")
print("01_Capa_Percepcion_V1.xlsx")


Archivo generado correctamente
01_Capa_Percepcion_V1.xlsx
